<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Table of Contents:

- [Imports](#imports)
- [1 - Data Importing and Preparation](#1---data-importing-and-preparation)
    - 1a) Importing
    - 1b) Data Preparation
- [2 - Models with Default Hyperparameter Settings](#2---models-with-default-hyperparameter-settings)
    - 2a) Create 4 Algorithms
    - 2b) Stat Analysis of Ordinal Encoded 4 Algorithms
    - 2c) Stat Analysis of OneHotEncoded 4 Algorithms
    - 2d) Discussion of 4 Algorithms
- [3 - Tuning Model Hyperparameters](#3---tuning-model-hyperparameters)
    - 3a) Optimization of Ridge Ordinal
    - 3b) Optimization of Ridge OneHot
    - 3c) Optimization of Lasso Ordinal
    - 3d) Optimization of Lasso OneHot
    - 3e) Optimization of Decision Tree Regressor Ordinal
    - 3f) Optimization of Decision Tree Regressor OneHot
- [4 - Model Selection](#4---model-selection)
    - 4a) Collect Model Performances
    - 4b) Discussion of Assignment

</div>

##### **IMPORTANT**

- The hyperlinks still do not work as intended similar to assignment 1.  Please use the dropdown buttons to make things easier to find if found to be helpful for you!

---

##### **ORIENTATION**

- This follows COMP4432's week 3-4 assignment that is a follow-on from assignment 1 where this assignment focuses on performance observations between different regression models and encoding types.
- Regression Models utilized
    - Linear Regression
    - Lasso
    - Ridge
    - Decision Tree Regressor
- Encoding Types
    - Onehot
    - Ordinal

---

##### **SUMMARY**

- Completed

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## Imports

- [Back to Table of Contents](#table-of-contents)

</div>

In [1]:
# Arrays and matricies
import numpy as np

# Databasing
import pandas as pd

# Visualizations
import matplotlib.pyplot as plt
import seaborn as sns

# Data splitting and CV
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV

# 4 Regression Type Models
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.tree import DecisionTreeRegressor

# Metrics
from sklearn.metrics import root_mean_squared_error, mean_absolute_error

# Encoders/Scalers/Transformers
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Ignore warnings because no...
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## 1 - Data Importing and Preparation

- [Back to Table of Contents](#table-of-contents)

</div>

#### **1a) Importing**

In [168]:
# Get the Seaborn Diamonds Dataset
df = sns.load_dataset("diamonds")

# Ensure intial df shape is the same as assignment 1
print(f'53940 Rows?\t{df.shape[0] == 53940}'
      f'\n10 Cols?\t{df.shape[1] == 10}')

53940 Rows?	True
10 Cols?	True


---

#### **1b) Data Preparation**

##### 1bPRE)

Recreate the same data preparation that was conducted in assignment 1 and ensure the resulting data frame is 53,784 rows and 10 columns

In [169]:
# Define correct label orders IAW instructions
true_cut_labels = ['Ideal', 'Premium', 'Very Good', 'Good', 'Fair']
true_color_labels = ['D', 'E', 'F', 'G', 'H', 'I', 'J']
true_clarity_labels = ['IF', 'VVS1', 'VVS2', 'VS1', 'VS2', 'SI1', 'SI2', 'I1']

# Define categorical/numerical features
categorical_features = df.columns[df.dtypes == 'category'].to_list()
numerical_features = df.drop(columns='price').columns[df.drop(columns='price').dtypes != 'category'].to_list()

In [170]:
# Handle zeros IAW instructions
df = df[(df["x"] != 0) & (df["y"] != 0)].reset_index(drop=True)

# Impute zeros in 'z' IAW instructions
z_imputed = (df["depth"] / 100) * ((df["x"] + df["y"]) / 2)
df["z"] = df["z"].where(df["z"] != 0, z_imputed)

# Filter to 1 <= x,y,z <= 11
df = df[
(df['x'].between(1,11)) & \
(df['y'].between(1,11)) & \
(df['z'].between(1,11)) \
]

# Drop duplicate rows
df = df.drop_duplicates()

# Ensure the prepared df is the same as assignment 1
print(f'Rows 53784?\t{df.shape[0] == 53784}\n'
      f'Cols 10?\t{df.shape[1] == 10}')

Rows 53784?	True
Cols 10?	True


In [171]:
# Keep this line for reproducibility of split dataframes WITHOUT storing them in memory
# Index 0 = X_train
# Index 1 = X_test
# Index 2 = y_train
# Index 3 = y_test
# (train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))

##### 1bi)

In [172]:
# Ensure ordinal encoder follows the same ranking schema defined earlier
# Define ordering IAW dataframe precedence
# NOTE: Encoding starts with 0 which should denote the worst, so orders will be flipped
cat_order = [
    list(reversed(true_cut_labels)),
    list(reversed(true_color_labels)),
    list(reversed(true_clarity_labels))
]

# Create nuanced OrdinalEncoder
ranked_ordenc = OrdinalEncoder(
    categories=cat_order,
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

In [173]:
# Use ColumnTransformer to transform categorical with OrdinalEncoder and numerical with StandardScaler
column_transformer_ordenc = ColumnTransformer(transformers=[
                                                    ("cat_transform", ranked_ordenc, categorical_features), 
                                                    ("num_transform", StandardScaler(), numerical_features)],
                                        remainder='passthrough',
                                        verbose_feature_names_out=False)

# Fit the data on the training dataset
column_transformer_ordenc.fit(X=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[0])

# Transform and rebuild training dataset
X_train_ordenc = column_transformer_ordenc.transform(X=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[0])
feature_names = column_transformer_ordenc.get_feature_names_out()
X_train_ordenc = pd.DataFrame(X_train_ordenc, columns=feature_names, index=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[0].index)

# Transform and rebuild test dataset
X_test_ordenc = column_transformer_ordenc.transform(X=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[1])
feature_names = column_transformer_ordenc.get_feature_names_out()
X_test_ordenc = pd.DataFrame(X_test_ordenc, columns=feature_names, index=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[1].index)

# Verify shape is 43027 x 9
print(f'Rows 43027?\t{X_train_ordenc.shape[0] == 43027}\n'
      f'Cols 9?\t\t{X_train_ordenc.shape[1] == 9}')

Rows 43027?	True
Cols 9?		True


##### 1bii)

In [174]:
# Create the correctly ordered OneHotEncoder
ranked_onehot = OneHotEncoder(
    categories=cat_order,
    sparse_output=False,
    drop='first'
)

In [175]:
# Use ColumnTransformer to transform categorical with OneHotEncoder and numerical with StandardScaler
column_transformer_onehot = ColumnTransformer(transformers=[
    ("cat_transform", ranked_onehot, categorical_features),
    ("num_transform", StandardScaler(), numerical_features)],
    remainder="passthrough",
    verbose_feature_names_out=False)

# Fit the data on the training dataset
column_transformer_onehot.fit(X=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[0])

# Transform and rebuild the training dataset
X_train_onehot = column_transformer_onehot.transform(X=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[0])
feature_names = column_transformer_onehot.get_feature_names_out()
X_train_onehot = pd.DataFrame(X_train_onehot, columns=feature_names, index=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[0].index)

# Transform and rebuild the testing dataset
X_test_onehot = column_transformer_onehot.transform(X=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[1])
feature_names = column_transformer_onehot.get_feature_names_out()
X_test_onehot = pd.DataFrame(X_test_onehot, columns=feature_names, index=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[1].index)

# Verify shape is 43027 x 23
print(f'Rows 43027?\t{X_train_onehot.shape[0] == 43027}\n'
      f'Cols 23?\t{X_train_onehot.shape[1] == 23}')

Rows 43027?	True
Cols 23?	True


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## 2 - Models with Default Hyperparameter Settings

- [Back to Table of Contents](#table-of-contents)

</div>

#### **2a) Create 4 Algorithms**

##### 2ai)

In [176]:
# Create the Ordinal/OneHot specific LinearRegressor models
lr_ordinal = LinearRegression()
lr_onehot = LinearRegression()

##### 2aii)

In [177]:
# Create the Ordinal/OneHot specific Lasso models IAW instructions
lasso_ordinal = Lasso(max_iter=10000, random_state=411)
lasso_onehot = Lasso(max_iter=10000, random_state=411)

##### 2aiii)

In [178]:
# Create the Ordinal/OneHot specific Ridge models IAW instructions
ridge_ordinal = Ridge(max_iter=10000, random_state=411)
ridge_onehot = Ridge(max_iter=10000, random_state=411)

##### 2aiv)

In [179]:
# Create the Ordinal/OneHot specific Decision Tree Regressor models IAW instructions
dtr_ordinal = DecisionTreeRegressor(random_state=411)
dtr_onehot = DecisionTreeRegressor(random_state=411)

---

#### **2b) Stat Analysis of Ordinal Encoded 4 Algorithms**

##### 2bi)

In [180]:
# Initial print for visibility assistance
print(f'\033[35mRMSE SCORES OF 5-FOLD ORDINAL\n{'-'*50}')

# Get the lr_ordinal RMSE scores and print info
lr_ordinal_rmse = cross_val_score(estimator=lr_ordinal,
                                  X=X_train_ordenc,
                                  y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                  cv=5,
                                  scoring='neg_root_mean_squared_error')
print(f'\033[36mlr_ordinal:\033[0m\t{abs(lr_ordinal_rmse)}')

# Get the lasso_ordinal RMSE scores and print info
lasso_ordinal_rmse = cross_val_score(estimator=lasso_ordinal,
                                     X=X_train_ordenc,
                                     y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                     cv=5,
                                     scoring='neg_root_mean_squared_error')
print(f'\033[36mlasso_ordinal:\033[0m\t{abs(lasso_ordinal_rmse)}')

# Get the ridge_ordinal RMSE scores and print info
ridge_ordinal_rmse = cross_val_score(estimator=ridge_ordinal,
                                     X=X_train_ordenc,
                                     y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                     cv=5,
                                     scoring='neg_root_mean_squared_error')
print(f'\033[36mridge_ordinal:\033[0m\t{abs(ridge_ordinal_rmse)}')

# Get the dtr_ordinal RMSE scores and print info
dtr_ordinal_rmse = cross_val_score(estimator=dtr_ordinal,
                                   X=X_train_ordenc,
                                   y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                   cv=5,
                                   scoring='neg_root_mean_squared_error')
print(f'\033[36mdtr_ordinal:\033[0m\t{abs(dtr_ordinal_rmse)}')

RMSE SCORES OF 5-FOLD ORDINAL
--------------------------------------------------
lr_ordinal:	[1165.56519438 1219.46863801 1210.73781263 1196.29946994 1234.26041429]


lasso_ordinal:	[1166.10543245 1219.58227291 1212.05287352 1198.14836522 1234.09898348]
ridge_ordinal:	[1165.54734151 1219.39180931 1210.76178899 1196.36952018 1234.18617816]
dtr_ordinal:	[722.8832774  738.76206455 773.7558487  731.05586241 739.68872585]


##### 2bii)

In [181]:
# Initial print for visibility assistance
print(f'\033[35mMEAN RMSE SCORES OF 5-FOLD ORDINAL\n{'-'*50}')

# Print off lr_ordinal
print(f'\033[36mlr_ordinal:\033[0m\t{float(abs(lr_ordinal_rmse.mean()))}')

# Print off lasso_ordinal
print(f'\033[36mlasso_ordinal:\033[0m\t{float(abs(lasso_ordinal_rmse.mean()))}')

# Print off ridge_ordinal
print(f'\033[36mridge_ordinal:\033[0m\t{float(abs(ridge_ordinal_rmse.mean()))}')

# Print off dtr_ordinal
print(f'\033[36mdtr_ordinal:\033[0m\t{float(abs(dtr_ordinal_rmse.mean()))}')

MEAN RMSE SCORES OF 5-FOLD ORDINAL
--------------------------------------------------
lr_ordinal:	1205.266305852553
lasso_ordinal:	1205.9975855157586
ridge_ordinal:	1205.2513276307677
dtr_ordinal:	741.2291557824705


##### 2biii)

In [182]:
# Initial print for visibility assistance
print(f'\033[35mSTD RMSE SCORES OF 5-FOLD ORDINAL\n{'-'*50}')

# Print off lr_ordinal
print(f'\033[36mlr_ordinal:\033[0m\t{float(lr_ordinal_rmse.std())}')

# Print off lasso_ordinal
print(f'\033[36mlasso_ordinal:\033[0m\t{float(lasso_ordinal_rmse.std())}')

# Print off ridge_ordinal
print(f'\033[36mridge_ordinal:\033[0m\t{float(ridge_ordinal_rmse.std())}')

# Print off dtr_ordinal
print(f'\033[36mdtr_ordinal:\033[0m\t{float(dtr_ordinal_rmse.std())}')

STD RMSE SCORES OF 5-FOLD ORDINAL
--------------------------------------------------
lr_ordinal:	23.361867850940705
lasso_ordinal:	23.081997086771736
ridge_ordinal:	23.33596794431349
dtr_ordinal:	17.358066523131544


##### 2biv)

In [183]:
# Initial print for visibility assistance
print(f'\033[35mMAE SCORES OF 5-FOLD ORDINAL\n{'-'*50}')

# Get the lr_ordinal MAE scores and print info
lr_ordinal_mae = cross_val_score(estimator=lr_ordinal,
                                 X=X_train_ordenc,
                                 y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                 cv=5,
                                 scoring='neg_mean_absolute_error')
print(f'\033[36mlr_ordinal:\033[0m\t{abs(lr_ordinal_mae)}')

# Get the lasso_ordinal MAE scores and print info
lasso_ordinal_mae = cross_val_score(estimator=lasso_ordinal,
                                    X=X_train_ordenc,
                                    y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                    cv=5,
                                    scoring='neg_mean_absolute_error')
print(f'\033[36mlasso_ordinal:\033[0m\t{abs(lasso_ordinal_mae)}')

# Get the ridge_ordinal MAE scores and print info
ridge_ordinal_mae = cross_val_score(estimator=ridge_ordinal,
                                    X=X_train_ordenc,
                                    y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                    cv=5,
                                    scoring='neg_mean_absolute_error')
print(f'\033[36mridge_ordinal:\033[0m\t{abs(ridge_ordinal_mae)}')

# Get the dtr_ordinal MAE scores and print info
dtr_ordinal_mae = cross_val_score(estimator=dtr_ordinal,
                                  X=X_train_ordenc,
                                  y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                  cv=5,
                                  scoring='neg_mean_absolute_error')
print(f'\033[36mdtr_ordinal:\033[0m\t{abs(dtr_ordinal_mae)}')

MAE SCORES OF 5-FOLD ORDINAL
--------------------------------------------------
lr_ordinal:	[786.21165439 798.26982978 816.31560276 796.34063189 795.79134898]
lasso_ordinal:	[786.60963012 798.59510531 816.00875432 797.44028072 795.25656971]
ridge_ordinal:	[786.24092442 798.28763802 816.29523365 796.42913615 795.77149073]
dtr_ordinal:	[359.24819893 365.64646758 376.05920976 358.46345148 364.51121441]


##### 2bv)

In [184]:
# Initial print for visibility assistance
print(f'\033[35mMEAN MAE SCORES OF 5-FOLD ORDINAL\n{'-'*50}')

# Print off lr_ordinal
print(f'\033[36mlr_ordinal:\033[0m\t{float(abs(lr_ordinal_mae.mean()))}')

# Print off lasso_ordinal
print(f'\033[36mlasso_ordinal:\033[0m\t{float(abs(lasso_ordinal_mae.mean()))}')

# Print off ridge_ordinal
print(f'\033[36mridge_ordinal:\033[0m\t{float(abs(ridge_ordinal_mae.mean()))}')

# Print off dtr_ordinal
print(f'\033[36mdtr_ordinal:\033[0m\t{float(abs(dtr_ordinal_mae.mean()))}')

MEAN MAE SCORES OF 5-FOLD ORDINAL
--------------------------------------------------
lr_ordinal:	798.5858135607398
lasso_ordinal:	798.782068034111
ridge_ordinal:	798.6048845946328
dtr_ordinal:	364.7857084330851


##### 2bvi)

In [185]:
# Initial print for visibility assistance
print(f'\033[35mSTD MAE SCORES OF 5-FOLD ORDINAL\n{'-'*50}')

# Print off lr_ordinal
print(f'\033[36mlr_ordinal:\033[0m\t{float(abs(lr_ordinal_mae.std()))}')

# Print off lasso_ordinal
print(f'\033[36mlasso_ordinal:\033[0m\t{float(abs(lasso_ordinal_mae.std()))}')

# Print off ridge_ordinal
print(f'\033[36mridge_ordinal:\033[0m\t{float(abs(ridge_ordinal_mae.std()))}')

# Print off dtr_ordinal
print(f'\033[36mdtr_ordinal:\033[0m\t{float(abs(dtr_ordinal_mae.std()))}')

STD MAE SCORES OF 5-FOLD ORDINAL
--------------------------------------------------
lr_ordinal:	9.802193213938542
lasso_ordinal:	9.583229643228895
ridge_ordinal:	9.784463278353376
dtr_ordinal:	6.301476492568505


---

#### **2c) Stat Analysis of OneHotEncoded 4 Algorithms**

##### 2ci)

In [186]:
# Initial print for visibility assistance
print(f'\033[35mRMSE SCORES OF 5-FOLD ONEHOT\n{'-'*50}')

# Get the lr_onehot RMSE scores and print info
lr_onehot_rmse = cross_val_score(estimator=lr_onehot,
                                 X=X_train_onehot,
                                 y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                 cv=5,
                                 scoring='neg_root_mean_squared_error')
print(f'\033[36mlr_onehot:\033[0m\t{abs(lr_onehot_rmse)}')

# Get the lasso_onehot RMSE scores and print info
lasso_onehot_rmse = cross_val_score(estimator=lasso_onehot,
                                    X=X_train_onehot,
                                    y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                    cv=5,
                                    scoring='neg_root_mean_squared_error')
print(f'\033[36mlasso_onehot:\033[0m\t{abs(lasso_onehot_rmse)}')

# Get the ridge_onehot RMSE scores and print info
ridge_onehot_rmse = cross_val_score(estimator=ridge_onehot,
                                    X=X_train_onehot,
                                    y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                    cv=5,
                                    scoring='neg_root_mean_squared_error')
print(f'\033[36mridge_onehot:\033[0m\t{abs(ridge_onehot_rmse)}')

# Get the dtr_onehot RMSE scores and print info
dtr_onehot_rmse = cross_val_score(estimator=dtr_onehot,
                                  X=X_train_onehot,
                                  y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                  cv=5,
                                  scoring='neg_root_mean_squared_error')
print(f'\033[36mdtr_onehot:\033[0m\t{abs(dtr_onehot_rmse)}')

RMSE SCORES OF 5-FOLD ONEHOT
--------------------------------------------------
lr_onehot:	[1089.50977703 1131.72877605 1117.40999402 1112.99538259 1154.43187445]
lasso_onehot:	[1090.65798782 1133.30466476 1121.47770874 1117.1999994  1157.67249038]
ridge_onehot:	[1089.33062228 1131.61620732 1117.56015689 1113.1112542  1154.54517439]
dtr_onehot:	[883.08505258 905.26101637 944.21147936 946.77257752 796.97108731]


##### 2cii)

In [187]:
# Initial print for visibility assistance
print(f'\033[35mMEAN RMSE SCORES OF 5-FOLD ONEHOT\n{'-'*50}')

# Print off lr_onehot
print(f'\033[36mlr_onehot:\033[0m\t{float(abs(lr_onehot_rmse.mean()))}')

# Print off lasso_onehot
print(f'\033[36mlasso_onehot:\033[0m\t{float(abs(lasso_onehot_rmse.mean()))}')

# Print off ridge_onehot
print(f'\033[36mridge_onehot:\033[0m\t{float(abs(ridge_onehot_rmse.mean()))}')

# Print off dtr_onehot
print(f'\033[36mdtr_onehot:\033[0m\t{float(abs(dtr_onehot_rmse.mean()))}')

MEAN RMSE SCORES OF 5-FOLD ONEHOT
--------------------------------------------------
lr_onehot:	1121.2151608292272
lasso_onehot:	1124.0625702188977
ridge_onehot:	1121.2326830154907
dtr_onehot:	895.260242625645


##### 2ciii)

In [188]:
# Initial print for visibility assistance
print(f'\033[35mSTD RMSE SCORES OF 5-FOLD ONEHOT\n{'-'*50}')

# Print off lr_onehot
print(f'\033[36mlr_onehot:\033[0m\t{float(lr_onehot_rmse.std())}')

# Print off lasso_onehot
print(f'\033[36mlasso_onehot:\033[0m\t{float(lasso_onehot_rmse.std())}')

# Print off ridge_onehot
print(f'\033[36mridge_onehot:\033[0m\t{float(ridge_onehot_rmse.std())}')

# Print off dtr_onehot
print(f'\033[36mdtr_onehot:\033[0m\t{float(dtr_onehot_rmse.std())}')

STD RMSE SCORES OF 5-FOLD ONEHOT
--------------------------------------------------
lr_onehot:	21.453025795483672
lasso_onehot:	21.838889885605408
ridge_onehot:	21.516162256464018
dtr_onehot:	54.696897544546815


##### 2civ)

In [189]:
# Initial print for visibility assistance
print(f'\033[35mMAE SCORES OF 5-FOLD ONEHOT\n{'-'*50}')

# Get the lr_onehot MAE scores and print info
lr_onehot_mae = cross_val_score(estimator=lr_onehot,
                                X=X_train_onehot,
                                y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                cv=5,
                                scoring='neg_mean_absolute_error')
print(f'\033[36mlr_onehot:\033[0m\t{abs(lr_onehot_mae)}')

# Get the lasso_onehot MAE scores and print info
lasso_onehot_mae = cross_val_score(estimator=lasso_onehot,
                                   X=X_train_onehot,
                                   y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                   cv=5,
                                   scoring='neg_mean_absolute_error')
print(f'\033[36mlasso_onehot:\033[0m\t{abs(lasso_onehot_mae)}')

# Get the ridge_onehot MAE scores and print info
ridge_onehot_mae = cross_val_score(estimator=ridge_onehot,
                                   X=X_train_onehot,
                                   y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                   cv=5,
                                   scoring='neg_mean_absolute_error')
print(f'\033[36mridge_onehot:\033[0m\t{abs(ridge_onehot_mae)}')

# Get the dtr_onehot MAE scores and print info
dtr_onehot_mae = cross_val_score(estimator=dtr_onehot,
                                 X=X_train_onehot,
                                 y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2],
                                 cv=5,
                                 scoring='neg_mean_absolute_error')
print(f'\033[36mdtr_onehot:\033[0m\t{abs(dtr_onehot_mae)}')

MAE SCORES OF 5-FOLD ONEHOT
--------------------------------------------------
lr_onehot:	[722.74348614 733.33568728 747.23368836 732.67029817 729.21028033]
lasso_onehot:	[717.99079999 728.6793261  742.20283254 729.35153243 725.11083663]
ridge_onehot:	[722.0720387  732.72000657 746.69582674 732.21652017 728.70302364]
dtr_onehot:	[399.3951894  408.22443644 419.11499128 410.70366066 383.75676932]


##### 2cv)

In [190]:
# Initial print for visibility assistance
print(f'\033[35mMEAN MAE SCORES OF 5-FOLD ONEHOT\n{'-'*50}')

# Print off lr_onehot
print(f'\033[36mlr_onehot:\033[0m\t{float(abs(lr_onehot_mae.mean()))}')

# Print off lasso_onehot
print(f'\033[36mlasso_onehot:\033[0m\t{float(abs(lasso_onehot_mae.mean()))}')

# Print off ridge_onehot
print(f'\033[36mridge_onehot:\033[0m\t{float(abs(ridge_onehot_mae.mean()))}')

# Print off dtr_onehot
print(f'\033[36mdtr_onehot:\033[0m\t{float(abs(dtr_onehot_mae.mean()))}')

MEAN MAE SCORES OF 5-FOLD ONEHOT
--------------------------------------------------
lr_onehot:	733.0386880568973
lasso_onehot:	728.667065538218
ridge_onehot:	732.4814831652043
dtr_onehot:	404.2390094218282


##### 2cvi)

In [191]:
# Initial print for visibility assistance
print(f'\033[35mSTD MAE SCORES OF 5-FOLD ONEHOT\n{'-'*50}')

# Print off lr_onehot
print(f'\033[36mlr_onehot:\033[0m\t{float(abs(lr_onehot_mae.std()))}')

# Print off lasso_onehot
print(f'\033[36mlasso_onehot:\033[0m\t{float(abs(lasso_onehot_mae.std()))}')

# Print off ridge_onehot
print(f'\033[36mridge_onehot:\033[0m\t{float(abs(ridge_onehot_mae.std()))}')

# Print off dtr_onehot
print(f'\033[36mdtr_onehot:\033[0m\t{float(abs(dtr_onehot_mae.std()))}')

STD MAE SCORES OF 5-FOLD ONEHOT
--------------------------------------------------
lr_onehot:	8.029568377514028
lasso_onehot:	7.878007915332171
ridge_onehot:	8.059875342719684
dtr_onehot:	12.01628006085365


---

#### **2d) Discussion of 4 Algorithms**

- **Discussion of 4 Algorithms Ordinal vs. Onehot**
    - Recall that ordinally encoded categorical features replace all unique values within one feature and the onehot encoding will separate every unique value as its own feature as a binary classification (i.e. T/F, 0/1)
    - Furthermore, though all 4 algorithms solve regression problems and generally attempt to minimize error and obtain a "best fit", linear regression, lasso, and ridge work best in the cleanest sense of regression in which a binary classification limits noise as opposed to the decision tree regressor where it has an easier time separating multi-class classifications.  In other words, it is expected that the former 3 algorithms will favor onehot transformations since every unique category is given a unique feature whereas the latter 1 algorithm will favor ordinal transforms all unique values within the already existing feature.
    - This being said, the above outputs reinforce this assumption where for the ordinally encoded data, the decision tree regressor performed the best by a fairly large margin in comparison to the other three and inversely for the onehot encoded data, the other three performed the best compared to the decision tree regressor.  It should also be noted that in both types of data encoding, the three regressors performed generally within ~1% of eachother demonstrating very little difference in performance between them.
- **Potential Next Steps**
    - Further distillation of which categorical features create the best information gain and specific values within the categorical features that create the best information gain in the case of creating the most lightweight yet high-performance predicitive model as possible
    - Taking the desired algorithm - likely the decision tree regressor with the ordinally encoded data - and refining the best combination of hyperparameters to pass into the model to ideally squeeze out any increase in performance if any exists
- **Early Stage Insight**
    - This reinforces the above assumption of what each regression model prefers in determining the "best fit" where multi-class is better for decision tree regressors and binary classes are better for linear regression, lasso, and ridge algorithms
    - The passed in *RANKED FROM LOWEST TO HIGHEST* categorical variables for training shows great information gain for predictions
- **Encoding Preference**
    - Ordinally encoded because the decision tree regressor still performed better as opposed to the onehot encoded 3 algorithms, especially since it is likely that the ranking of the multi-class categories was shown to have strong relevance to the target feature during exploration of assignment 1

<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## 3 - Tuning Model Hyperparameters

- [Back to Table of Contents](#table-of-contents)

</div>

#### **3a) Optimization of Ridge Ordinal**

##### 3ai-3aiv)

In [192]:
# Define Ridge hyperparameters for search
param_grid = {'alpha': [0.01, 5, 100],
              'max_iter': [None, 100, 10000],
              'tol': [0.000001, 0.00001, 0.0001, 0.001]}

# Setup GridSearchCV
ridge_grid_search_ordenc = GridSearchCV(estimator=ridge_ordinal,
                                        param_grid=param_grid,
                                        cv=5,
                                        verbose=0,
                                        scoring='neg_root_mean_squared_error')

# Execute GridSearchCV
ridge_grid_search_ordenc.fit(X=X_train_ordenc,
                             y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2])

# Print best score and parameters
print(f'\033[35mBest Score:\033[0m\t\t{abs(ridge_grid_search_ordenc.best_score_)}\n'
      f'\033[35mBest Hyperparameters:\033[0m\t{ridge_grid_search_ordenc.best_params_}')

Best Score:		1205.2649115465642
Best Hyperparameters:	{'alpha': 5, 'max_iter': None, 'tol': 1e-06}


---

#### **3b) Optimization of Ridge OneHot**

##### 3bi-3biv)

In [193]:
# Define Ridge hyperparameters for search
param_grid = {'alpha': [0.01, 5, 100],
              'max_iter': [None, 100, 10000],
              'tol': [0.000001, 0.00001, 0.0001, 0.001]}

# Setup GridSearchCV
ridge_grid_search_onehot = GridSearchCV(estimator=ridge_onehot,
                                        param_grid=param_grid,
                                        cv=5,
                                        verbose=0,
                                        scoring='neg_root_mean_squared_error')

# Execute GridSearchCV
ridge_grid_search_onehot.fit(X=X_train_onehot,
                             y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2])

# Print best score and parameters
print(f'\033[35mBest Score:\033[0m\t\t{abs(ridge_grid_search_onehot.best_score_)}\n'
      f'\033[35mBest Hyperparameters:\033[0m\t{ridge_grid_search_onehot.best_params_}')

Best Score:		1121.2150358862098
Best Hyperparameters:	{'alpha': 0.01, 'max_iter': None, 'tol': 1e-06}


---

#### **3c) Optimization of Lasso Ordinal**

##### 3ci-3civ)

In [194]:
# Define Lasso hyperparameters for search
param_grid = {'alpha': [0.01, 5, 100],
              'max_iter': [10000, 100000],
              'tol': [0.000001, 0.00001, 0.0001, 0.001]}

# Setup GridSearchCV
lasso_grid_search_ordenc = GridSearchCV(estimator=lasso_ordinal,
                                        param_grid=param_grid,
                                        cv=5,
                                        verbose=0,
                                        scoring='neg_root_mean_squared_error')

# Execute GridSearchCV
lasso_grid_search_ordenc.fit(X=X_train_ordenc,
                             y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2])

# Print best score and parameters
print(f'\033[35mBest Score:\033[0m\t\t{abs(lasso_grid_search_ordenc.best_score_)}\n'
      f'\033[35mBest Hyperparameters:\033[0m\t{lasso_grid_search_ordenc.best_params_}')

Best Score:		1205.2637491180963
Best Hyperparameters:	{'alpha': 0.01, 'max_iter': 10000, 'tol': 1e-06}


---

#### **3d) Optimization of Lass OneHot**

##### 3di-3div)

In [195]:
# Define Lasso hyperparameters for search
param_grid = {'alpha': [0.01, 5, 100],
              'max_iter': [10000, 100000],
              'tol': [0.000001, 0.00001, 0.0001, 0.001]}

# Setup GridSearchCV
lasso_grid_search_onehot = GridSearchCV(estimator=lasso_onehot,
                                        param_grid=param_grid,
                                        cv=5,
                                        verbose=0,
                                        scoring='neg_root_mean_squared_error')

# Execute GridSearchCV
lasso_grid_search_onehot.fit(X=X_train_onehot,
                             y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2])

# Print best score and parameters
print(f'\033[35mBest Score:\033[0m\t\t{abs(lasso_grid_search_onehot.best_score_)}\n'
      f'\033[35mBest Hyperparameters:\033[0m\t{lasso_grid_search_onehot.best_params_}')

Best Score:		1121.2135547887679
Best Hyperparameters:	{'alpha': 0.01, 'max_iter': 10000, 'tol': 1e-06}


---

#### **3e) Optimization of Decision Tree Regressor Ordinal**

##### 3ePRE)

In [196]:
# Determine max_depth for GridSearchCV
temp_dtr=DecisionTreeRegressor(random_state=411)

temp_dtr.fit(X=X_train_ordenc,
             y=train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411)[2])

temp_dtr.get_depth()

34

##### 3ei-3eiv)

In [197]:
# Define Decision Tree Regressor hyperparameters for search
param_grid = {'max_depth': list(range(34))}

# Setup GridSearchCV
dtr_grid_search_ordenc = GridSearchCV(estimator=dtr_ordinal,
                                        param_grid=param_grid,
                                        cv=10,
                                        verbose=0,
                                        scoring='neg_root_mean_squared_error')

# Execute GridSearchCV
dtr_grid_search_ordenc.fit(X=X_train_ordenc,
                             y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2])

# Print best score and parameters
print(f'\033[35mBest Score:\033[0m\t\t{abs(dtr_grid_search_ordenc.best_score_)}\n'
      f'\033[35mBest Hyperparameters:\033[0m\t{dtr_grid_search_ordenc.best_params_}')

Best Score:		637.1637223535065
Best Hyperparameters:	{'max_depth': 11}


/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
10 fits failed out of a total of 340.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py", line 436, in _validate_params

---

#### **3f) Optimization of Decision Tree Regressor OneHot**

##### 3fPRE)

In [198]:
# Determine max_depth for GridSearchCV
temp_dtr=DecisionTreeRegressor(random_state=411)

temp_dtr.fit(X=X_train_onehot,
             y=train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411)[2])

temp_dtr.get_depth()

38

##### 3fi-3fiv)

In [199]:
# Define Decision Tree Regressor hyperparameters for search
param_grid = {'max_depth': list(range(38))}

# Setup GridSearchCV
dtr_grid_search_onehot = GridSearchCV(estimator=dtr_onehot,
                                        param_grid=param_grid,
                                        cv=10,
                                        verbose=0,
                                        scoring='neg_root_mean_squared_error')

# Execute GridSearchCV
dtr_grid_search_onehot.fit(X=X_train_onehot,
                             y=(train_test_split(df.drop(columns='price'), df[['price']], test_size=0.20, random_state=411))[2])

# Print best score and parameters
print(f'\033[35mBest Score:\033[0m\t\t{abs(dtr_grid_search_onehot.best_score_)}\n'
      f'\033[35mBest Hyperparameters:\033[0m\t{dtr_grid_search_onehot.best_params_}')

/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
10 fits failed out of a total of 380.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
10 fits failed with the following error:
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py", line 1382, in wrapper
    estimator._validate_params()
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/opt/anaconda3/lib/python3.13/site-packages/sklearn/base.py", line 436, in _validate_params

Best Score:		875.7404053989388
Best Hyperparameters:	{'max_depth': 13}


<div style="
    padding: 15px 20px;
    margin: 10px 0;
    border-left: 8px solid #4F7942;
    background-color: rgba(255, 191, 0, 0.05);
    border-radius: 4px;
">

## 4 - Model Selection

- [Back to Table of Contents](#table-of-contents)

</div>

#### **4a) Collect Model Performances**

##### 4aPRE)

In [200]:
# Create a dictionary to store information then display IAW instructions
model_performance_dict = {
    'Model': [],
    'Encoder': [],
    'MeanRMSE': [],
    'Params': []
}

##### 4ai)

In [201]:
# Add the lr, lasso, ridge, dtr default ordinal models
# lr info
meanRMSE = float(abs(lr_ordinal_rmse.mean()))
params = lr_ordinal.get_params()
model_performance_dict['Model'].append('Default_LinearRegression')
model_performance_dict['Encoder'].append('Ordinal')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

# lasso info
meanRMSE = float(abs(lasso_ordinal_rmse.mean()))
params = lasso_ordinal.get_params()
model_performance_dict['Model'].append('Default_Lasso')
model_performance_dict['Encoder'].append('Ordinal')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

# ridge info
meanRMSE = float(abs(ridge_ordinal_rmse.mean()))
params = ridge_ordinal.get_params()
model_performance_dict['Model'].append('Default_Ridge')
model_performance_dict['Encoder'].append('Ordinal')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

# dtr info
meanRMSE = float(abs(dtr_ordinal_rmse.mean()))
params = lr_ordinal.get_params()
model_performance_dict['Model'].append('Default_DecisionTreeRegressor')
model_performance_dict['Encoder'].append('Ordinal')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

##### 4aii)

In [202]:
# Add the lr, lasso, ridge, dtr default onehot models
# lr info
meanRMSE = float(abs(lr_onehot_rmse.mean()))
params = lr_onehot.get_params()
model_performance_dict['Model'].append('Default_LinearRegression')
model_performance_dict['Encoder'].append('Onehot')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

# lasso info
meanRMSE = float(abs(lasso_onehot_rmse.mean()))
params = lasso_onehot.get_params()
model_performance_dict['Model'].append('Default_Lasso')
model_performance_dict['Encoder'].append('Onehot')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

# ridge info
meanRMSE = float(abs(ridge_onehot_rmse.mean()))
params = ridge_onehot.get_params()
model_performance_dict['Model'].append('Default_Ridge')
model_performance_dict['Encoder'].append('Onehot')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

# dtr info
meanRMSE = float(abs(dtr_onehot_rmse.mean()))
params = lr_onehot.get_params()
model_performance_dict['Model'].append('Default_DecisionTreeRegressor')
model_performance_dict['Encoder'].append('Onehot')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

##### 4aiii)

In [203]:
# Add the ridge ordinal optimal model
meanRMSE = float(abs(ridge_grid_search_ordenc.best_score_))
params = ridge_grid_search_ordenc.best_params_
model_performance_dict['Model'].append('Tuned_Ridge')
model_performance_dict['Encoder'].append('Ordinal')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

##### 4aiv)

In [204]:
# Add the ridge onehot optimal model
meanRMSE = float(abs(ridge_grid_search_onehot.best_score_))
params = ridge_grid_search_onehot.best_params_
model_performance_dict['Model'].append('Tuned_Ridge')
model_performance_dict['Encoder'].append('Onehot')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

##### 4av)

In [205]:
# Add the lasso ordinal optimal model
meanRMSE = float(abs(lasso_grid_search_ordenc.best_score_))
params = lasso_grid_search_ordenc.best_params_
model_performance_dict['Model'].append('Tuned_Lasso')
model_performance_dict['Encoder'].append('Ordinal')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

##### 4avi)

In [206]:
# Add the lasso onehot optimal model
meanRMSE = float(abs(lasso_grid_search_onehot.best_score_))
params = lasso_grid_search_onehot.best_params_
model_performance_dict['Model'].append('Tuned_Lasso')
model_performance_dict['Encoder'].append('Onehot')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

##### 4avii)

In [207]:
# Add the dtr ordinal optimal model
meanRMSE = float(abs(dtr_grid_search_ordenc.best_score_))
params = dtr_grid_search_ordenc.best_params_
model_performance_dict['Model'].append('Tuned_DecisionTreeRegressor')
model_performance_dict['Encoder'].append('Ordinal')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

##### 4aviii)

In [208]:
# Add the dtr onehot optimal model
meanRMSE = float(abs(dtr_grid_search_onehot.best_score_))
params = dtr_grid_search_onehot.best_params_
model_performance_dict['Model'].append('Tuned_DecisionTreeRegressor')
model_performance_dict['Encoder'].append('Onehot')
model_performance_dict['MeanRMSE'].append(meanRMSE)
model_performance_dict['Params'].append(params)

##### 4aPOST)

In [209]:
# Transfer the compiled data as a Pandas dataframe
model_performance_df = pd.DataFrame(model_performance_dict)

# Print off initial information
model_performance_df

,Model,Encoder,MeanRMSE,Params
0,Default_LinearRegression,Ordinal,1205.266306,"{'copy_X': True, 'fit_intercept': True, 'n_job..."
1,Default_Lasso,Ordinal,1205.997586,"{'alpha': 1.0, 'copy_X': True, 'fit_intercept'..."
2,Default_Ridge,Ordinal,1205.251328,"{'alpha': 1.0, 'copy_X': True, 'fit_intercept'..."
3,Default_DecisionTreeRegressor,Ordinal,741.229156,"{'copy_X': True, 'fit_intercept': True, 'n_job..."
4,Default_LinearRegression,Onehot,1121.215161,"{'copy_X': True, 'fit_intercept': True, 'n_job..."
5,Default_Lasso,Onehot,1124.062570,"{'alpha': 1.0, 'copy_X': True, 'fit_intercept'..."
6,Default_Ridge,Onehot,1121.232683,"{'alpha': 1.0, 'copy_X': True, 'fit_intercept'..."
7,Default_DecisionTreeRegressor,Onehot,895.260243,"{'copy_X': True, 'fit_intercept': True, 'n_job..."
8,Tuned_Ridge,Ordinal,1205.264912,"{'alpha': 5, 'max_iter': None, 'tol': 1e-06}"
9,Tuned_Ridge,Onehot,1121.215036,"{'alpha': 0.01, 'max_iter': None, 'tol': 1e-06}"


In [210]:
# Replicate the dataframe IAW instructions
model_performance_df.drop(columns=['Params']).sort_values(by='MeanRMSE', ascending=False).reset_index(drop=True)

,Model,Encoder,MeanRMSE
0,Default_Lasso,Ordinal,1205.997586
1,Default_LinearRegression,Ordinal,1205.266306
2,Tuned_Ridge,Ordinal,1205.264912
3,Tuned_Lasso,Ordinal,1205.263749
4,Default_Ridge,Ordinal,1205.251328
5,Default_Lasso,Onehot,1124.062570
6,Default_Ridge,Onehot,1121.232683
7,Default_LinearRegression,Onehot,1121.215161
8,Tuned_Ridge,Onehot,1121.215036
9,Tuned_Lasso,Onehot,1121.213555


---

#### **4b) Discussion of Assignment**

- Was regularization helpful or an improvement to regression modeling? What does this imply for linear regression?
    - Regularization was a general improvement from default models where the models where the tuned versions performed slightly lower than the default likely because the GridSearchCV hyperparameters to search through was very thin to limit run-time.  This assignment demonstrates how further regularization and intentional decisions of what features to encode, how, and what machine models to select for the best possible performance is necessary for better model predicitions.

- How did encoding the categorical features impact model performance? Discuss whether any encoding method resulted in better predictability.
    - As mentioned earlier, Onehot encoding will separate every unique value in a categorical feature in to as many binary features as necessary whereas Ordinal encoding will convert every unique value into a numerical value within the categorical feature for interpretability with machines.  Alternatively, Onehot makes observations of binary relationships more clear and Ordinal will make observations of multi-variate relationships more clear.  That being said, Linear Regression, Lasso, and Ridge are examples of models that are closely related to the idea of "regression" in the sense of a best fit line in relation to the data.  In which case, it's clear why encoding styles like Onehot is best for these models because it isolates all possible binary relationships for easy best fit line determination.  On the other hand, Decision Tree Regressor is best for multi-variable relationships which - yet again - would explain why the Ordinal encoding helped the Decision Tree Regressor perform better as opposed to the Onehot encoding style.

- With the findings from Part 4(a), what can be further gleaned from the data after further model development? Refer to Parts 3(a) and 3(b) of Assignment 1.  While a simple graph, the observed vertical and horizontal variances are helpful evidence.
    - From assignment 1, the exploratory analysis with simple models and various input features and encoding types gleaned how meaningful features directly impact model performance as it relates to the target feature;  Alternatively, the more related the input feature is to the target feature, the more of a positive impact is expected to be observed.  Near the end of assignment 1, it was demonstrated how more meaningful input data will reduce the vertical variance in model predictive power as well as scaling and encoding input data.  Furthermore, the horizontal distribution of the data become more normalized through the encoding.  This assignment, assignment 2, focused more on manipulation of different **types** of regression models like Linear Regression, Ridge, Lasso, and Decision Tree Regressor ran with Ordinal/Onehot encoded categorical features and default/tuned hyperparameters in relation to the specific regression model.  Unlike assignment 1, this showed a clear distinction in how the models are impacted by encoding methodologies where the best method is the one that goes in-line with the models intended method of minimalizing the normal equation (i.e. Linear Regression, Lasso, and Ridge prefer binary relations so Onehot encoding is best; whereas Decision Tree Regressor prefer multi-variate relations so Ordinal encoding is best).  This emphasizes how there's never a singular best solution to any problem in the field of Data Science and rather needs to be meticulously curated in relation to the input data, what the input means in relation to the target, decisions on dealing with categorical features, and what model would best fit the optimization for accurate predictions of the target variable.

- Which of the above models should be selected as the champion model?
    - "Tuned_DecisionTreeRegressor" // Ordinal // 637.16